# Regression to the Mean: Extremes Drift Back

## Before/after stories are tempting — extreme baselines make them dangerous

## Takeaway

Functions introduced: `sort_values`, `corr`, `pd.to_datetime`.

**Concept learned: extreme selection can make ordinary drift look causal.**

Run the cell below first. It enlarges the font for both code and markdown so the notebook is easy to read while walking through it in class.

In [ ]:
from IPython.display import HTML, display

display(HTML("""
<style>
/* Rendered markdown */
.jp-RenderedHTMLCommon,
.jp-RenderedMarkdown,
.rendered_html {
    font-size: 24px !important;
    line-height: 1.5 !important;
}
.jp-RenderedHTMLCommon h1, .rendered_html h1 { font-size: 40px !important; }
.jp-RenderedHTMLCommon h2, .rendered_html h2 { font-size: 34px !important; }
.jp-RenderedHTMLCommon h3, .rendered_html h3 { font-size: 30px !important; }
.jp-RenderedHTMLCommon h4, .rendered_html h4 { font-size: 28px !important; }
.jp-RenderedHTMLCommon table, .rendered_html table {
    font-size: 22px !important;
}

/* Code editor (CodeMirror, used by classic + JupyterLab) */
.CodeMirror, .cm-editor, .jp-Editor, .jp-InputArea-editor {
    font-size: 24px !important;
}
.cm-content, .cm-line { font-size: 24px !important; }

/* Code output (print, tracebacks, DataFrame text) */
.jp-OutputArea-output,
.output_area,
.output pre,
.jp-RenderedText pre {
    font-size: 22px !important;
}

/* DataFrame tables in output */
.dataframe, .dataframe th, .dataframe td {
    font-size: 22px !important;
}
</style>
"""))


### Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['font.size'] = 16
plt.rcParams['figure.figsize'] = (8, 5)

## The story

Students with very low baseline scores tend to improve on a follow-up test, even with no intervention. Students with very high baseline scores tend to drop. This is **regression to the mean** — extremes drift back toward average just because of measurement noise.

## 1. Load the score dataset

Rows are students with baseline, follow-up, and change.

In [ ]:
scores = pd.read_csv("../data/regression_to_mean_scores.csv")
scores.head()

In [ ]:
scores.info()

## 2. Find extremes with `df.sort_values()`

Sorting reveals the selected extremes.

In [ ]:
lowest = scores.sort_values("baseline_score").head(20)
lowest

In [ ]:
highest = scores.sort_values("baseline_score", ascending=False).head(20)
highest

## 3. Compare change in the extreme groups

If regression to the mean is at play, the lowest baselines should *rise* on follow-up and the highest baselines should *fall* — even without any intervention.

In [ ]:
low = scores.sort_values("baseline_score").head(100)
high = scores.sort_values("baseline_score").tail(100)
print(f"Avg change for 100 lowest baselines:  {low['change'].mean():+.2f}")
print(f"Avg change for 100 highest baselines: {high['change'].mean():+.2f}")

## 4. Build a both-tails group

Concatenate the two extreme tails to compare against the middle.

In [ ]:
extremes = pd.concat([
    scores.sort_values("baseline_score").head(60),
    scores.sort_values("baseline_score").tail(60),
])
extremes.shape

## 5. Visualize the drift

In [ ]:
ax = scores.plot(kind="scatter", x="baseline_score", y="change",
                 alpha=0.3, title="Change vs. baseline score")
ax.axhline(0, color="red", linestyle="--")
plt.show()

Low baselines mostly rise; high baselines mostly fall. The line of zero change cuts diagonally through the cloud.

## 6. Relationships with `df.corr()`

Correlation helps describe the link between baseline, follow-up, and change.

In [ ]:
scores[["baseline_score", "followup_score", "change"]].corr()

## 7. Change scores need suspicion

`change` is strongly negatively correlated with `baseline` almost automatically: baseline includes random noise that subtracts out in the change.

In [ ]:
scores[["baseline_score", "change"]].corr()

## 8. Dates with `pd.to_datetime()`

Before/after analyses often require real datetime columns. We do not have a date here, but the same technique applies on the churn dataset.

In [ ]:
churn = pd.read_csv("../data/misleading_variables_churn.csv")
churn["signup_date"] = pd.to_datetime(churn["signup_date"])
churn["signup_date"].dtype

## 9. Check the type after conversion

In [ ]:
churn[["signup_date"]].dtypes

## Mini-lab: extremes drift

In [ ]:
low = scores.sort_values("baseline_score").head(100)
high = scores.sort_values("baseline_score").tail(100)
print("Low baseline avg change:", round(low["change"].mean(), 2))
print("High baseline avg change:", round(high["change"].mean(), 2))

## Discussion

- If the lowest-scoring students improved after coaching, what else must be true before claiming the intervention worked?
- What would a fair comparison group look like?

**Real-world examples:** bad sales months rebound, career-best athletes decline, angry customers calm down, extreme stores normalize.